# GurobiPy - End-to-End Learning Notebook
### From Zero to Production | Free Web License Edition

> **License note:** This notebook uses Gurobi free Web License (WLS) which supports models up to  
> **2,000 variables x 2,000 constraints**. No grbgetkey or license file required.

---

## Table of Contents
1. [Setup and Installation](#1)
2. [Module 1 - Core Fundamentals](#2)
3. [Module 2 - Data Structures and Expressions](#3)
4. [Module 3 - Problem Classes](#4)
5. [Module 4 - Diagnostics and Troubleshooting](#5)
6. [Module 5 - Advanced Features](#6)
7. [Module 6 - Expert Extras and Cheat Sheet](#7)


---
<a id="1"></a>
## Section 1 - Setup and Installation


In [ ]:
# Install gurobipy (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gurobipy", "-q"])
print("gurobipy installed")

In [ ]:
# Verify install and confirm free web license works
import gurobipy as gp
from gurobipy import GRB

print(f"Gurobi version: {gp.gurobi.version()}")

with gp.Env() as env:
    env.setParam("OutputFlag", 0)
    with gp.Model(env=env) as m:
        x = m.addVar(lb=0)
        m.setObjective(x, GRB.MINIMIZE)
        m.addConstr(x >= 5)
        m.optimize()
        print(f"License OK: solved trivial LP, x* = {x.X}")

print("""
Free Web License limits:
  - Variables  : 2,000
  - Constraints: 2,000
  - All model types supported: LP / MILP / QP / MIQP
  All examples in this notebook stay within these limits.
""")

---
<a id="2"></a>
## Module 1 - Core Fundamentals

### Core Architecture

Every GurobiPy program follows this hierarchy:

    gp.Model() -> addVar/addVars -> setObjective -> addConstr/addConstrs -> optimize()

### Variable Types

| Constant | Type | Use |
|---|---|---|
| GRB.CONTINUOUS | Real | LP (default) |
| GRB.INTEGER | Integer | MILP |
| GRB.BINARY | 0 or 1 | Decision variables |

### Key Model Status Codes

| Constant | Value | Meaning |
|---|---|---|
| GRB.OPTIMAL | 2 | Optimal solution found |
| GRB.INFEASIBLE | 3 | No feasible solution |
| GRB.UNBOUNDED | 5 | Objective is unbounded |
| GRB.TIME_LIMIT | 9 | Time limit hit |
| GRB.SUBOPTIMAL | 13 | Best solution found, not proven optimal |


### 1.1 Your First LP - Minimal Working Example

**Problem:** Maximize profit from two products

    maximize    5*x1 + 4*x2
    subject to   x1 +   x2 <= 40   (resource A)
                2*x1 +  x2 <= 60   (resource B)
                 x1,    x2 >= 0


In [ ]:
import gurobipy as gp
from gurobipy import GRB

# "with" context manager auto-calls m.dispose() on exit -> no memory leaks
with gp.Model("first_lp") as m:
    m.setParam("OutputFlag", 1)   # 1=show log, 0=silent

    # --- Decision Variables ---
    # lb=0 and vtype=CONTINUOUS are defaults; shown here for clarity
    x1 = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name="x1")
    x2 = m.addVar(lb=0, vtype=GRB.CONTINUOUS, name="x2")

    # --- Objective Function ---
    # Python operators (+, *, <=) are overloaded to build Gurobi LinExpr objects
    m.setObjective(5*x1 + 4*x2, GRB.MAXIMIZE)

    # --- Constraints ---
    c_A = m.addConstr(x1 + x2  <= 40, name="resource_A")
    c_B = m.addConstr(2*x1 + x2 <= 60, name="resource_B")

    # --- Solve ---
    m.optimize()

    # --- Extract Results ---
    if m.Status == GRB.OPTIMAL:
        print(f"
Optimal Objective = {m.ObjVal:.4f}")
        print(f"x1 = {x1.X:.4f}")
        print(f"x2 = {x2.X:.4f}")
        print(f"
Shadow Prices (objective change per unit of RHS):")
        print(f"  resource_A: Pi = {c_A.Pi:.4f}  Slack = {c_A.Slack:.4f}")
        print(f"  resource_B: Pi = {c_B.Pi:.4f}  Slack = {c_B.Slack:.4f}")
    elif m.Status == GRB.INFEASIBLE:
        print("Model is infeasible")
    elif m.Status == GRB.UNBOUNDED:
        print("Model is unbounded")

---
<a id="3"></a>
## Module 2 - Data Structures and Expressions

### 2.1 gp.tuplelist - Fast Arc Selection


In [ ]:
import gurobipy as gp

# gp.tuplelist: list of tuples with fast .select() method
arcs = gp.tuplelist([
    ("A", "X"), ("A", "Y"),
    ("B", "X"), ("B", "Z"),
    ("C", "Y"), ("C", "Z"),
])

# .select() uses wildcard "*" - much faster than list comprehensions
from_A = arcs.select("A", "*")   # all arcs leaving A
to_X   = arcs.select("*", "X")   # all arcs arriving at X

print(f"Arcs from A : {list(from_A)}")
print(f"Arcs to X   : {list(to_X)}")
print(f"Total arcs  : {len(arcs)}")

### 2.2 gp.tupledict - Variables over Index Sets

addVars() returns a **tupledict** which provides:
- .select("A", "*") - partial index filtering
- .sum("*", "X")    - build LinExpr over matching keys
- .prod(coeff_dict) - fastest weighted sum (pure C++ call)


In [ ]:
import gurobipy as gp
from gurobipy import GRB

arcs = gp.tuplelist([
    ("A","X"),("A","Y"),("B","X"),("B","Z"),("C","Y"),("C","Z")
])
costs = {("A","X"):2.5, ("A","Y"):3.0, ("B","X"):1.8,
         ("B","Z"):2.2, ("C","Y"):4.1, ("C","Z"):1.5}

with gp.Model("tupledict_demo") as m:
    m.setParam("OutputFlag", 0)

    # addVars(tuplelist) -> tupledict of Var objects
    flow = m.addVars(arcs, lb=0, name="flow")

    print(f"Type of flow : {type(flow)}")
    print(f"Keys         : {list(flow.keys())}")

    # .sum() builds a LinExpr across matching keys
    out_A = flow.sum("A", "*")    # flow[A,X] + flow[A,Y]
    in_X  = flow.sum("*", "X")    # flow[A,X] + flow[B,X]
    print(f"
out_A expr: {out_A}")

    # .prod() is the FASTEST way to build a weighted sum (no Python loop)
    obj = flow.prod(costs)
    m.setObjective(obj, GRB.MINIMIZE)
    m.addConstr(flow.sum("A","*") == 10, "supply_A")
    m.addConstr(flow.sum("*","X") ==  8, "demand_X")
    m.optimize()
    print(f"
Min cost = {m.ObjVal:.2f}")

### 2.3 quicksum vs sum - Performance


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import time

# Rule: always use gp.quicksum() for Gurobi expressions
# Standard sum() creates N intermediate LinExpr objects -> quadratic memory

N = 500  # kept small for the free license

with gp.Model() as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(N, name="x")
    coeffs = list(range(N))

    # SLOW: Python built-in sum
    t0 = time.perf_counter()
    slow = sum(coeffs[i] * x[i] for i in range(N))
    t1 = time.perf_counter()

    # FAST: gp.quicksum
    t2 = time.perf_counter()
    fast = gp.quicksum(coeffs[i] * x[i] for i in range(N))
    t3 = time.perf_counter()

    # FASTEST: tupledict.prod (pure C++)
    coeff_dict = {i: coeffs[i] for i in range(N)}
    t4 = time.perf_counter()
    fastest = x.prod(coeff_dict)
    t5 = time.perf_counter()

    print(f"{'Method':<25} {'Time (ms)':>10}")
    print("-" * 37)
    print(f"{'sum()':<25} {(t1-t0)*1000:>10.3f}")
    print(f"{'gp.quicksum()':<25} {(t3-t2)*1000:>10.3f}")
    print(f"{'tupledict.prod()':<25} {(t5-t4)*1000:>10.3f}")
    print("
Prefer: prod() > quicksum() > sum()")

### 2.4 Multi-Dimensional Indices - 3-Echelon Supply Chain


In [ ]:
import gurobipy as gp
from gurobipy import GRB

plants     = ["P1", "P2"]
warehouses = ["W1", "W2", "W3"]
customers  = ["C1", "C2", "C3", "C4"]

supply      = {"P1": 500, "P2": 400}
demand      = {"C1": 150, "C2": 200, "C3": 130, "C4": 220}
wh_capacity = {"W1": 300, "W2": 250, "W3": 200}

cost_pw = {("P1","W1"):2, ("P1","W2"):3, ("P1","W3"):4,
           ("P2","W1"):3, ("P2","W2"):2, ("P2","W3"):1}

cost_wc = {("W1","C1"):3, ("W1","C2"):5, ("W1","C4"):6,
           ("W2","C1"):4, ("W2","C2"):2, ("W2","C3"):3,
           ("W3","C3"):2, ("W3","C4"):4}

with gp.Model("supply_chain") as m:
    m.setParam("OutputFlag", 0)

    f_pw = m.addVars(cost_pw.keys(), lb=0, name="plant_to_wh")
    f_wc = m.addVars(cost_wc.keys(), lb=0, name="wh_to_cust")

    m.setObjective(f_pw.prod(cost_pw) + f_wc.prod(cost_wc), GRB.MINIMIZE)

    m.addConstrs((f_pw.sum(p,"*") <= supply[p]      for p in plants),      name="supply")
    m.addConstrs((f_pw.sum("*",w) <= wh_capacity[w] for w in warehouses),  name="wh_cap")
    m.addConstrs((f_pw.sum("*",w) == f_wc.sum(w,"*") for w in warehouses), name="balance")
    m.addConstrs(
        (f_wc.sum("*",c) >= demand[c]
         for c in customers if any((w,c) in cost_wc for w in warehouses)),
        name="demand"
    )
    m.optimize()

    if m.Status == GRB.OPTIMAL:
        print(f"Min shipping cost: ${m.ObjVal:,.2f}
")
        print("Plant to Warehouse:")
        for (p,w),v in f_pw.items():
            if v.X > 0.01: print(f"  {p} -> {w}: {v.X:.1f}")
        print("Warehouse to Customer:")
        for (w,c),v in f_wc.items():
            if v.X > 0.01: print(f"  {w} -> {c}: {v.X:.1f}")

---
<a id="4"></a>
## Module 3 - Problem Classes and Case Studies

### 3a. LP - Diet / Blending Problem

**Formulation:**

    min   sum_j  c_j * x_j                      (minimize cost)
    s.t.  sum_j  a_ij * x_j >= b_i_min   for i  (minimum nutrient)
          sum_j  a_ij * x_j <= b_i_max   for i  (maximum nutrient)
          x_j >= 0


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

foods     = ["chicken", "eggs", "milk", "bread", "spinach", "pasta"]
nutrients = ["protein", "fat", "calories", "iron", "calcium"]

cost = {"chicken":3.19, "eggs":0.25, "milk":0.89,
        "bread":0.15, "spinach":1.89, "pasta":0.45}

nutrition = {
    ("chicken","protein"):60, ("chicken","fat"):20, ("chicken","calories"):295,
    ("chicken","iron"):1.8,  ("chicken","calcium"):20,
    ("eggs","protein"):13,   ("eggs","fat"):10,    ("eggs","calories"):155,
    ("eggs","iron"):2.0,     ("eggs","calcium"):56,
    ("milk","protein"):8,    ("milk","fat"):5,     ("milk","calories"):122,
    ("milk","iron"):0.1,     ("milk","calcium"):291,
    ("bread","protein"):3,   ("bread","fat"):1,    ("bread","calories"):65,
    ("bread","iron"):0.6,    ("bread","calcium"):26,
    ("spinach","protein"):5, ("spinach","fat"):0,  ("spinach","calories"):23,
    ("spinach","iron"):6.4,  ("spinach","calcium"):245,
    ("pasta","protein"):8,   ("pasta","fat"):1,    ("pasta","calories"):220,
    ("pasta","iron"):2.0,    ("pasta","calcium"):12,
}

reqs = {
    "protein":  [50,  150],
    "fat":      [20,   70],
    "calories": [1800, 2500],
    "iron":     [8,    45],
    "calcium":  [1000, 2500],
}

with gp.Model("diet") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(foods, lb=0, ub=10, name="servings")
    m.setObjective(x.prod(cost), GRB.MINIMIZE)

    m.addConstrs(
        (gp.quicksum(nutrition.get((f,n),0) * x[f] for f in foods) >= reqs[n][0]
         for n in nutrients), name="min_nut"
    )
    m.addConstrs(
        (gp.quicksum(nutrition.get((f,n),0) * x[f] for f in foods) <= reqs[n][1]
         for n in nutrients), name="max_nut"
    )
    m.optimize()

    if m.Status == GRB.OPTIMAL:
        print(f"Min daily food cost: ${m.ObjVal:.2f}
")
        df = pd.DataFrame({
            "Food":      foods,
            "Servings":  [round(x[f].X, 3) for f in foods],
            "Cost":      [f"${cost[f]*x[f].X:.2f}" for f in foods],
        })
        df = df[df["Servings"] > 0.001].reset_index(drop=True)
        print(df.to_string(index=False))
        print("
Nutrient Check:")
        for n in nutrients:
            total = sum(nutrition.get((f,n),0)*x[f].X for f in foods)
            print(f"  {n:10s}: {total:7.1f}  (range: {reqs[n][0]} to {reqs[n][1]})")

### 3b. MILP - Facility Location Problem

**Formulation:**

    min   sum_i  f_i*y_i  +  sum_i sum_j  c_ij*x_ij
    s.t.  sum_i  x_ij = 1             for all j  (each customer served)
          x_ij <= y_i                 for all i,j (only open facilities serve)
          y_i in {0,1},  x_ij in {0,1}


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import random

random.seed(42)
I = list(range(8))    # candidate facility locations
J = list(range(25))   # customers  -> 25*8 = 200 binary vars (within free limit)

fixed_cost     = {i: random.randint(100, 500) for i in I}
transport_cost = {(i,j): random.randint(1, 50) for i in I for j in J}

with gp.Model("facility_location") as m:
    m.setParam("OutputFlag", 1)
    m.setParam("MIPGap",    0.01)
    m.setParam("TimeLimit", 30)

    y = m.addVars(I, vtype=GRB.BINARY, name="open")
    x = m.addVars(I, J, vtype=GRB.BINARY, name="assign")

    m.setObjective(y.prod(fixed_cost) + x.prod(transport_cost), GRB.MINIMIZE)

    # Every customer must be served by exactly one facility
    m.addConstrs((x.sum("*", j) == 1 for j in J), name="serve_all")

    # Can only assign from open facilities (STRONG formulation)
    m.addConstrs((x[i,j] <= y[i] for i in I for j in J), name="link")

    m.optimize()

    if m.Status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        opened = [i for i in I if y[i].X > 0.5]
        print(f"
Total cost    : {m.ObjVal:,.2f}")
        print(f"MIP Gap       : {m.MIPGap*100:.3f}%")
        print(f"Nodes explored: {int(m.NodeCount)}")
        print(f"Open facilities: {opened}")
        for i in opened:
            served = [j for j in J if x[i,j].X > 0.5]
            print(f"  Facility {i} serves customers: {served}")

### 3c. QP - Portfolio Optimization (Markowitz Mean-Variance)

**Formulation:**

    min   x'Sigma x                (minimize portfolio variance)
    s.t.  mu'x >= r_target          (minimum expected return)
          sum_i x_i = 1              (fully invested)
          x_i >= 0                   (no short-selling)


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

tickers = ["AAPL", "GOOGL", "MSFT", "JPM", "XOM"]
n = len(tickers)

mu = np.array([0.18, 0.15, 0.16, 0.10, 0.08])

sigma = np.array([
    [0.040, 0.010, 0.015, 0.005, 0.002],
    [0.010, 0.030, 0.012, 0.004, 0.001],
    [0.015, 0.012, 0.035, 0.006, 0.002],
    [0.005, 0.004, 0.006, 0.020, 0.008],
    [0.002, 0.001, 0.002, 0.008, 0.015],
])

def solve_portfolio(target_return):
    with gp.Model("portfolio") as m:
        m.setParam("OutputFlag", 0)
        x = m.addVars(n, lb=0, ub=1, name="w")

        # Quadratic objective: minimize x'Sigma x
        variance = gp.quicksum(
            sigma[i,j] * x[i] * x[j]
            for i in range(n) for j in range(n)
        )
        m.setObjective(variance, GRB.MINIMIZE)

        m.addConstr(gp.quicksum(x[i] for i in range(n)) == 1, "budget")
        m.addConstr(gp.quicksum(mu[i]*x[i] for i in range(n)) >= target_return,
                    "return_target")
        m.optimize()

        if m.Status == GRB.OPTIMAL:
            w = [x[i].X for i in range(n)]
            var = m.ObjVal
            ret = sum(mu[i]*w[i] for i in range(n))
            return {
                "Target": target_return,
                "Return": round(ret, 4),
                "StdDev": round(np.sqrt(var), 4),
                "Sharpe": round((ret - 0.03) / np.sqrt(var), 3),
                "Weights": dict(zip(tickers, [round(wi,4) for wi in w])),
            }
    return None

# Trace efficient frontier
targets  = np.linspace(0.09, 0.17, 15)
frontier = [r for t in targets if (r := solve_portfolio(t)) is not None]

df = pd.DataFrame(frontier).drop(columns=["Weights"])
print("Efficient Frontier:")
print(df.to_string(index=False))

best = max(frontier, key=lambda r: r["Sharpe"])
print(f"
Max Sharpe Portfolio (Sharpe = {best['Sharpe']:.3f})")
print(f"  Return : {best['Return']*100:.2f}%")
print(f"  StdDev : {best['StdDev']*100:.2f}%")
for t,w in best["Weights"].items():
    if w > 0.01: print(f"  {t}: {w*100:.1f}%")

---
<a id="5"></a>
## Module 4 - Diagnostics and Troubleshooting

### 4.1 Querying Solution Attributes

| Attribute | Object | Meaning | When |
|---|---|---|---|
| .X | Var | Solution value | After OPTIMAL |
| .RC | Var | Reduced cost | LP only |
| .VarName | Var | Variable name | Always |
| .Pi | Constr | Shadow price | LP only |
| .Slack | Constr | Constraint slack | After OPTIMAL |
| .IISConstr | Constr | In IIS? | After computeIIS() |
| m.ObjVal | Model | Objective value | After OPTIMAL |
| m.MIPGap | Model | Relative gap | After MILP |
| m.Runtime | Model | Solve time (s) | After optimize() |


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

with gp.Model("inspect") as m:
    m.setParam("OutputFlag", 0)
    x1 = m.addVar(lb=0, name="x1")
    x2 = m.addVar(lb=0, name="x2")
    m.setObjective(5*x1 + 4*x2, GRB.MAXIMIZE)
    c1 = m.addConstr(x1 + x2   <= 40, "resource_A")
    c2 = m.addConstr(2*x1 + x2 <= 60, "resource_B")
    m.optimize()

    if m.Status == GRB.OPTIMAL:
        print(f"Objective  : {m.ObjVal:.4f}")
        print(f"Solve time : {m.Runtime:.4f}s
")

        var_df = pd.DataFrame([
            {"Name": v.VarName, "Value": v.X, "Reduced Cost": v.RC}
            for v in m.getVars()
        ])
        print("Variables:")
        print(var_df.to_string(index=False))

        con_df = pd.DataFrame([
            {"Name": c.ConstrName, "Shadow Price": c.Pi,
             "Slack": c.Slack, "Binding": abs(c.Slack) < 1e-6}
            for c in m.getConstrs()
        ])
        print("
Constraints:")
        print(con_df.to_string(index=False))

        print("
Interpretation:")
        for c in m.getConstrs():
            if abs(c.Pi) > 1e-8:
                print(f"  {c.ConstrName} shadow price = {c.Pi:.2f}")
                print(f"  -> Each additional unit adds ${c.Pi:.2f} to objective")

        m.write("/tmp/solution.sol")
        m.write("/tmp/model.lp")
        print("
Files written: /tmp/solution.sol  /tmp/model.lp")

### 4.2 IIS - Irreducible Inconsistent Subsystem

IIS finds the **minimum set of constraints that conflict** when a model is infeasible.


In [ ]:
import gurobipy as gp
from gurobipy import GRB

with gp.Model("infeasible") as m:
    m.setParam("OutputFlag", 0)

    x = m.addVar(lb=0, ub=5, name="x")   # x <= 5
    y = m.addVar(lb=0, ub=5, name="y")   # y <= 5
    m.setObjective(x + y, GRB.MINIMIZE)

    # Conflict: x+y >= 12 is impossible when x<=5 and y<=5
    m.addConstr(x + y >= 12, "sum_min")
    m.addConstr(x <= 5,      "x_max")
    m.addConstr(y <= 5,      "y_max")
    m.optimize()

    if m.Status == GRB.INFEASIBLE:
        print("Model is INFEASIBLE
")

        # Method 1: computeIIS -> find conflicting constraints
        m.computeIIS()
        print("IIS Constraints (the conflict):")
        for c in m.getConstrs():
            if c.IISConstr: print(f"  CONFLICT: {c.ConstrName}")
        print("
IIS Variable Bounds:")
        for v in m.getVars():
            if v.IISLB: print(f"  CONFLICT: {v.VarName} lower bound {v.LB}")
            if v.IISUB: print(f"  CONFLICT: {v.VarName} upper bound {v.UB}")
        m.write("/tmp/iis_report.ilp")

        # Method 2: feasRelax -> find nearest feasible point
        print("
Computing feasibility relaxation...")
        orig_vars = m.NumVars
        m.feasRelax(0, False, False, True)   # 0=min violations, relax constraints
        m.optimize()
        print(f"  Minimum total violation: {m.ObjVal:.4f} units")
        for v in m.getVars()[orig_vars:]:
            if v.X > 1e-6:
                print(f"  {v.VarName} needs relaxation of {v.X:.4f}")

### 4.3 Sensitivity Analysis - Dual Values and Shadow Prices

Shadow price = how much the objective changes per unit increase in the RHS.


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

with gp.Model("sensitivity") as m:
    m.setParam("OutputFlag", 0)

    x1 = m.addVar(lb=0, name="product_A")
    x2 = m.addVar(lb=0, name="product_B")
    m.setObjective(5*x1 + 4*x2, GRB.MAXIMIZE)

    c1 = m.addConstr(x1 + x2   <= 40, "labor_hours")
    c2 = m.addConstr(2*x1 + x2 <= 60, "machine_hours")
    c3 = m.addConstr(x1        <= 30, "product_A_cap")
    m.optimize()

    print(f"Optimal profit: ${m.ObjVal:.2f}
")

    rows = []
    for c in m.getConstrs():
        rows.append({
            "Constraint":   c.ConstrName,
            "Shadow Price": round(c.Pi, 4),
            "Slack":        round(c.Slack, 4),
            "Binding":      abs(c.Slack) < 1e-6,
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

    print("
Interpretation:")
    print(f"  labor_hours shadow price = {c1.Pi:.2f}")
    print(f"  -> Each extra labor hour is worth ${c1.Pi:.2f} in profit")
    print(f"  -> Worth paying up to ${c1.Pi:.2f}/hour for overtime")
    print(f"  machine_hours shadow price = {c2.Pi:.2f}")
    print(f"  -> Each extra machine hour is worth ${c2.Pi:.2f}")
    print(f"  product_A_cap slack = {c3.Slack:.2f} -> not binding")

    print("
Reduced Costs:")
    for v in m.getVars():
        print(f"  {v.VarName:<15} X={v.X:.4f}  RC={v.RC:.4f}")

---
<a id="6"></a>
## Module 5 - Advanced Features

### 5.1 Multi-Objective Optimization

| Strategy | When to Use | How |
|---|---|---|
| Lexicographic | Order matters: obj 1 must be optimal first | Different priority values |
| Blended | Trade-off acceptable | Same priority, different weights |


In [ ]:
import gurobipy as gp
from gurobipy import GRB

workers = [0, 1, 2]
tasks   = [0, 1, 2]
cost_matrix = [[3,5,2],[4,2,6],[1,3,4]]
time_matrix = [[2,4,1],[3,1,5],[2,2,3]]

print("STRATEGY 1: LEXICOGRAPHIC")
with gp.Model("lex") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(workers, tasks, vtype=GRB.BINARY, name="assign")
    m.addConstrs((x.sum(w,"*") == 1 for w in workers), "one_task")
    m.addConstrs((x.sum("*",t) == 1 for t in tasks),  "one_worker")

    # Priority 2 = optimized FIRST (higher number = higher priority)
    m.setObjectiveN(
        gp.quicksum(cost_matrix[w][t]*x[w,t] for w in workers for t in tasks),
        index=0, priority=2, weight=1.0, name="cost"
    )
    # Priority 1 = optimized SECOND
    m.setObjectiveN(
        gp.quicksum(time_matrix[w][t]*x[w,t] for w in workers for t in tasks),
        index=1, priority=1, weight=1.0, name="time"
    )
    m.ModelSense = GRB.MINIMIZE
    m.optimize()

    if m.Status == GRB.OPTIMAL:
        for w in workers:
            for t in tasks:
                if x[w,t].X > 0.5:
                    print(f"  Worker {w} -> Task {t}")
        m.ObjNumber = 0; print(f"  Total cost: {m.ObjNVal:.0f}")
        m.ObjNumber = 1; print(f"  Total time: {m.ObjNVal:.0f}")

print("
STRATEGY 2: BLENDED (0.6 cost + 0.4 time)")
with gp.Model("blend") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(workers, tasks, vtype=GRB.BINARY, name="assign")
    m.addConstrs((x.sum(w,"*") == 1 for w in workers), "one_task")
    m.addConstrs((x.sum("*",t) == 1 for t in tasks),  "one_worker")

    # Same priority -> combined as weighted sum
    m.setObjectiveN(
        gp.quicksum(cost_matrix[w][t]*x[w,t] for w in workers for t in tasks),
        index=0, priority=1, weight=0.6, name="cost"
    )
    m.setObjectiveN(
        gp.quicksum(time_matrix[w][t]*x[w,t] for w in workers for t in tasks),
        index=1, priority=1, weight=0.4, name="time"
    )
    m.ModelSense = GRB.MINIMIZE
    m.optimize()
    if m.Status == GRB.OPTIMAL:
        for w in workers:
            for t in tasks:
                if x[w,t].X > 0.5:
                    print(f"  Worker {w} -> Task {t}")
        m.ObjNumber = 0; print(f"  Cost: {m.ObjNVal:.0f}")
        m.ObjNumber = 1; print(f"  Time: {m.ObjNVal:.0f}")

### 5.2 Callbacks - Lazy Constraints (TSP Subtour Elimination)

Callbacks inject custom logic into Branch and Bound.  
Lazy constraints are only added when violated by an integer solution.


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import math, random

random.seed(7)
n = 10   # 10 cities -> 45 binary edge vars, well within free limit
coords = {i: (random.uniform(0,100), random.uniform(0,100)) for i in range(n)}
cities = list(range(n))

def dist(i, j):
    xi,yi = coords[i]; xj,yj = coords[j]
    return math.sqrt((xi-xj)**2 + (yi-yj)**2)

def find_subtour(vals):
    adj = {i: [] for i in cities}
    for i in cities:
        for j in range(i+1, n):
            if vals[i,j] > 0.5:
                adj[i].append(j); adj[j].append(i)
    visited = set(); best = None
    for start in cities:
        if start in visited: continue
        tour = []; cur = start; prev = -1
        while True:
            tour.append(cur); visited.add(cur)
            nxt = [v for v in adj[cur] if v != prev and v not in set(tour[:-1])]
            if not nxt: break
            prev, cur = cur, nxt[0]
        if best is None or len(tour) < len(best): best = tour
    return best or list(cities)

with gp.Model("TSP") as m:
    m.setParam("OutputFlag", 1)
    m.setParam("LazyConstraints", 1)   # REQUIRED for cbLazy()

    x = m.addVars(
        [(i,j) for i in range(n) for j in range(i+1,n)],
        vtype=GRB.BINARY, name="edge"
    )
    m.setObjective(
        gp.quicksum(dist(i,j)*x[i,j] for i in range(n) for j in range(i+1,n)),
        GRB.MINIMIZE
    )

    def city_edges(i):
        return [x[min(i,j), max(i,j)] for j in cities if j != i]

    m.addConstrs(
        (gp.quicksum(city_edges(i)) == 2 for i in cities), name="degree"
    )

    def subtour_cb(model, where):
        if where == GRB.Callback.MIPSOL:
            vals = model.cbGetSolution(x)
            tour = find_subtour(vals)
            if len(tour) < n:
                S = set(tour)
                cuts = [x[min(i,j),max(i,j)] for i in S for j in S if i < j]
                model.cbLazy(gp.quicksum(cuts) <= len(S) - 1)

    m.optimize(subtour_cb)

    if m.Status == GRB.OPTIMAL:
        print(f"
Optimal TSP tour length: {m.ObjVal:.2f}")
        vals = m.getAttr("X", x)
        tour = find_subtour(vals)
        print(f"Tour: {tour + [tour[0]]}")

### 5.3 Parameter Tuning Reference


In [ ]:
import gurobipy as gp
from gurobipy import GRB

with gp.Model("params") as m:
    # Output
    m.setParam("OutputFlag",     0)      # 0=silent, 1=verbose
    m.setParam("LogFile",       "")      # write log to file
    m.setParam("DisplayInterval", 5)    # log every N seconds

    # Termination
    m.setParam("TimeLimit",    120)     # max seconds
    m.setParam("MIPGap",      0.005)    # stop at 0.5% gap
    m.setParam("MIPGapAbs",   0.01)    # or absolute gap
    m.setParam("SolutionLimit",  5)    # stop after N integer solutions

    # Performance
    m.setParam("Threads",      4)      # CPU threads
    m.setParam("Presolve",     2)      # -1=auto 0=off 1=conservative 2=aggressive
    m.setParam("Cuts",         2)      # cut aggressiveness: -1 to 3
    m.setParam("Heuristics",  0.2)    # fraction of time for heuristics

    # LP Method: -1=auto 0=primal 1=dual 2=barrier 3=concurrent
    m.setParam("Method",       2)

    # MIP Focus: 0=balanced 1=feasible 2=optimal 3=bound
    m.setParam("MIPFocus",     1)

    # Numerical stability
    m.setParam("NumericFocus",   1)     # 0=speed to 3=max precision
    m.setParam("FeasibilityTol", 1e-8)
    m.setParam("OptimalityTol",  1e-9)
    m.setParam("IntFeasTol",     1e-6)

    # Callbacks
    m.setParam("LazyConstraints", 1)

    # Save / load params
    m.write("/tmp/params.prm")
    m.read("/tmp/params.prm")
    print("Parameter reference cell - no model solved")
    print("Params saved to /tmp/params.prm")

### 5.4 Production Architecture - Clean Solver Pattern


In [ ]:
# Production OOP pattern: one class per problem, structured result
import gurobipy as gp
from gurobipy import GRB
from dataclasses import dataclass, field
import logging

logger = logging.getLogger(__name__)

@dataclass
class OptResult:
    status:        str
    objective:     object = None
    solution:      object = None
    solve_time:    object = None
    mip_gap:       object = None
    error_message: object = None
    iis_constraints: list = field(default_factory=list)

class KnapsackOptimizer:

    STATUS_MAP = {
        GRB.OPTIMAL:    "optimal",
        GRB.INFEASIBLE: "infeasible",
        GRB.UNBOUNDED:  "unbounded",
        GRB.TIME_LIMIT: "time_limit",
        GRB.SUBOPTIMAL: "suboptimal",
    }

    def solve(self, values, weights, capacity, time_limit=30.0):
        try:
            return self._solve(values, weights, capacity, time_limit)
        except gp.GurobiError as e:
            logger.error(f"GurobiError: {e}")
            return OptResult(status="error", error_message=str(e))
        except Exception as e:
            return OptResult(status="error", error_message=str(e))

    def _solve(self, values, weights, capacity, time_limit):
        n = len(values)
        with gp.Model("knapsack") as m:
            m.setParam("OutputFlag",  0)
            m.setParam("TimeLimit", time_limit)

            x = m.addVars(n, vtype=GRB.BINARY, name="item")
            m.setObjective(gp.quicksum(values[i]*x[i] for i in range(n)), GRB.MAXIMIZE)
            m.addConstr(gp.quicksum(weights[i]*x[i] for i in range(n)) <= capacity, "cap")
            m.optimize()

            status = self.STATUS_MAP.get(m.Status, f"code_{m.Status}")
            if m.Status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
                selected = [i for i in range(n) if x[i].X > 0.5]
                return OptResult(
                    status=status, objective=round(m.ObjVal,4),
                    solve_time=round(m.Runtime,4),
                    solution={
                        "selected": selected,
                        "value":    sum(values[i]  for i in selected),
                        "weight":   sum(weights[i] for i in selected),
                    }
                )
            if m.Status == GRB.INFEASIBLE:
                m.computeIIS()
                return OptResult(status=status,
                    iis_constraints=[c.ConstrName for c in m.getConstrs() if c.IISConstr])
            return OptResult(status=status)

# Demo
items   = ["laptop","camera","headphones","tablet","charger","book"]
values  = [300, 150, 80, 200, 30, 10]
weights = [4.0, 1.5, 0.5, 1.2, 0.3, 0.2]

solver = KnapsackOptimizer()
result = solver.solve(values, weights, capacity=5.0)

print(f"Status    : {result.status}")
print(f"Objective : ${result.objective}")
print(f"Time      : {result.solve_time}s")
if result.solution:
    sel = result.solution["selected"]
    print(f"Selected  : {[items[i] for i in sel]}")
    print(f"Value     : ${result.solution['value']}")
    print(f"Weight    : {result.solution['weight']} kg")

---
<a id="7"></a>
## Module 6 - Expert Extras

### 6.1 Warm Starts and Solution Hints


In [ ]:
import gurobipy as gp
from gurobipy import GRB

items   = list(range(8))
values  = [10, 6, 5, 4, 3, 2, 1, 1]
weights = [ 5, 4, 3, 2, 1, 1, 1, 1]
cap     = 8

# Cold solve
with gp.Model("cold") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(items, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(values[i]*x[i] for i in items), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(weights[i]*x[i] for i in items) <= cap, "cap")
    m.optimize()
    print(f"Cold: obj={m.ObjVal:.0f}  nodes={int(m.NodeCount)}  time={m.Runtime:.4f}s")
    hint = [round(x[i].X) for i in items]

# Warm solve
with gp.Model("warm") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVars(items, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(values[i]*x[i] for i in items), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(weights[i]*x[i] for i in items) <= cap, "cap")

    # Set .Start BEFORE optimize() to provide warm start
    for i in items: x[i].Start = hint[i]

    m.optimize()
    print(f"Warm: obj={m.ObjVal:.0f}  nodes={int(m.NodeCount)}  time={m.Runtime:.4f}s")

print("
Warm starts help most when:")
print("  - Re-solving with slightly changed data (rolling horizon)")
print("  - Providing a heuristic feasible solution to B&B")
print("  - Search space is large and finding feasibility is hard")

### 6.2 General Constraints


In [ ]:
import gurobipy as gp
from gurobipy import GRB

# Absolute value: z = |x|
with gp.Model("abs") as m:
    m.setParam("OutputFlag", 0)
    x = m.addVar(lb=-10, ub=10, name="x")
    z = m.addVar(lb=0,          name="z")
    m.addGenConstrAbs(z, x)
    m.setObjective(z, GRB.MINIMIZE)
    m.addConstr(x == -7.5)
    m.optimize()
    print(f"abs(-7.5) = {z.X:.1f}")

# Max: c = max(a, b)
with gp.Model("max_constr") as m:
    m.setParam("OutputFlag", 0)
    a = m.addVar(lb=0, ub=10); b = m.addVar(lb=0, ub=10)
    c = m.addVar(lb=0)
    m.addGenConstrMax(c, [a, b])
    m.setObjective(c, GRB.MINIMIZE)
    m.addConstr(a == 3); m.addConstr(b == 7)
    m.optimize()
    print(f"max(3, 7) = {c.X:.1f}")

# Indicator: if y=1 then x <= 20
with gp.Model("indicator") as m:
    m.setParam("OutputFlag", 0)
    y = m.addVar(vtype=GRB.BINARY, name="y")
    x = m.addVar(lb=0, ub=100,    name="x")
    m.addGenConstrIndicator(y, True, x <= 20)
    m.setObjective(x, GRB.MAXIMIZE)
    m.addConstr(y == 1)   # force indicator ON
    m.optimize()
    print(f"max x given y=1 (x<=20): {x.X:.1f}")

# Piecewise linear: f(t) with breakpoints
with gp.Model("pwl") as m:
    m.setParam("OutputFlag", 0)
    t  = m.addVar(lb=0, ub=10, name="t")
    pw = m.addVar(lb=0,        name="f")
    x_pts = [0, 2, 5, 8, 10]
    y_pts = [0, 6, 9, 14, 16]
    m.addGenConstrPWL(t, pw, x_pts, y_pts)
    m.setObjective(pw, GRB.MAXIMIZE)
    m.optimize()
    print(f"max PWL at t={t.X:.1f}: f={pw.X:.1f}")

print("
General constraints model non-linearities without Big-M tricks.")

### 6.3 SOS Constraints


In [ ]:
import gurobipy as gp
from gurobipy import GRB

# SOS1: at most ONE variable non-zero -> "choose one option"
print("SOS1: select one product line")
with gp.Model("sos1") as m:
    m.setParam("OutputFlag", 0)
    revenue = [100, 250, 180, 320]
    budget  = [20,  60,  40,  80]
    x = m.addVars(4, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="frac")
    m.addSOS(GRB.SOS_TYPE1, [x[i] for i in range(4)])
    m.setObjective(gp.quicksum(revenue[i]*x[i] for i in range(4)), GRB.MAXIMIZE)
    m.addConstr(gp.quicksum(budget[i]*x[i] for i in range(4)) <= 70, "budget")
    m.optimize()
    chosen = [i for i in range(4) if x[i].X > 0.01]
    print(f"  Best option: {chosen}  Revenue={m.ObjVal:.0f}")

# SOS2: at most TWO consecutive variables non-zero -> piecewise linear
print("
SOS2: piecewise linear interpolation")
with gp.Model("sos2") as m:
    m.setParam("OutputFlag", 0)
    bp_x = [0, 2, 4, 6, 8, 10]
    bp_y = [0, 3, 5, 6, 8,  9]
    lam = m.addVars(len(bp_x), lb=0, name="lambda")
    y   = m.addVar(lb=0, name="output")
    m.addSOS(GRB.SOS_TYPE2, [lam[i] for i in range(len(bp_x))],
             weights=list(range(len(bp_x))))
    m.addConstr(gp.quicksum(lam[i] for i in range(len(bp_x))) == 1,  "convex")
    m.addConstr(gp.quicksum(bp_y[i]*lam[i] for i in range(len(bp_x))) == y, "interp")
    m.setObjective(y, GRB.MAXIMIZE)
    m.optimize()
    x_val = sum(bp_x[i]*lam[i].X for i in range(len(bp_x)))
    print(f"  Optimal t={x_val:.2f}  f(t)={y.X:.2f}")

---
## Complete GurobiPy Cheat Sheet


In [ ]:
# ================================================================
#  GUROBIPY COMPLETE CHEAT SHEET
#  Copy cells below as needed - no model is solved here
# ================================================================
import gurobipy as gp
from gurobipy import GRB

# -- Model Lifecycle --------------------------------------------------
m = gp.Model("name")            # create
m.setParam("OutputFlag", 0)     # silence
m.update()                      # sync lazy changes (usually auto)
m.optimize()                    # solve
m.reset()                       # clear solution, keep structure
m.dispose()                     # free memory  <-- use "with" instead
m.copy()                        # deep copy
m.write("f.lp")                 # export: .lp .mps .sol .prm .ilp
m.read("f.sol")                 # load solution or params

# -- Variable Creation ------------------------------------------------
v  = m.addVar(lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name="v")
vs = m.addVars(I, J, lb=0, vtype=GRB.BINARY, name="x")   # -> tupledict
# vtypes: GRB.CONTINUOUS  GRB.INTEGER  GRB.BINARY  GRB.SEMICONT

# -- Expressions (fastest to slowest) ---------------------------------
e1 = vs.prod(coeff_dict)                    # FASTEST: pure C++ call
e2 = gp.quicksum(c[i]*vs[i] for i in I)    # FAST: use instead of sum()
e3 = vs.sum("*", j)                         # partial tupledict sum
# Never use: sum(c[i]*vs[i] for i in I)    # SLOW: quadratic memory

# -- Objective --------------------------------------------------------
m.setObjective(expr, GRB.MINIMIZE)
m.setObjective(expr, GRB.MAXIMIZE)
m.setObjectiveN(expr, idx, priority, weight, name="obj")   # multi-obj
m.ModelSense = GRB.MINIMIZE

# -- Constraints ------------------------------------------------------
c1 = m.addConstr(lhs <= rhs, name="c")
c2 = m.addConstr(lhs == rhs, name="eq")
cs = m.addConstrs((e[i] <= b[i] for i in I), name="cs")
m.addRange(expr, lo, hi, name="range")
m.addQConstr(qexpr <= rhs, name="qc")
m.addSOS(GRB.SOS_TYPE1, vars)
m.addSOS(GRB.SOS_TYPE2, vars, weights)
m.addGenConstrIndicator(y, True, lhs <= rhs)
m.addGenConstrAbs(z, x)
m.addGenConstrMax(z, [a, b])
m.addGenConstrMin(z, [a, b])
m.addGenConstrPWL(xvar, yvar, xpts, ypts)

# -- Solution Attributes ----------------------------------------------
# m.Status       2=Optimal 3=Infeasible 5=Unbounded 9=TimeLimit 13=Suboptimal
# m.ObjVal       objective value
# m.ObjBound     best dual bound (MILP)
# m.MIPGap       |ObjVal-ObjBound|/|ObjVal|
# m.Runtime      seconds
# m.NodeCount    B&B nodes explored
# v.X            solution value      <- most used!
# v.RC           reduced cost (LP)
# v.Start        warm start hint
# c.Pi           shadow price / dual (LP)
# c.Slack        constraint slack
# c.IISConstr    in IIS? (after computeIIS())

# -- Key Parameters ---------------------------------------------------
# "OutputFlag"      0/1           silence or verbose
# "TimeLimit"       seconds       termination
# "MIPGap"          0.01          1% optimality gap
# "Threads"         int           CPU parallelism (-1=all)
# "MIPFocus"        0/1/2/3       balanced/feasible/optimal/bound
# "Presolve"        -1/0/1/2      auto/off/conservative/aggressive
# "Cuts"            -1 to 3       cut aggressiveness
# "Method"          -1/0/1/2/3    auto/primal/dual/barrier/concurrent
# "NumericFocus"    0/1/2/3       speed vs precision
# "LazyConstraints" 1             enable cbLazy() in callback
# "DualReductions"  0             set this if you get INF_OR_UNBD

# -- Infeasibility Tools ----------------------------------------------
# m.computeIIS()
# m.write("debug.ilp")
# iis = [c.ConstrName for c in m.getConstrs() if c.IISConstr]
# m.feasRelax(0, False, False, True)   # relax constraints

# -- Top 5 Pitfalls ---------------------------------------------------
# 1. Never call v.X before optimize()          -> AttributeError
# 2. Never use sum() for Gurobi expressions    -> slow + memory waste
# 3. Forget dispose()                          -> use "with" context manager
# 4. Read results on INFEASIBLE status         -> always check m.Status first
# 5. Get INF_OR_UNBD (code 4)                 -> set DualReductions=0 and re-solve

print("Cheat sheet loaded - no model solved")
print("Reference this cell anytime you need the API surface")

---

## What is Next?

| Topic | Resource |
|---|---|
| More examples | https://www.gurobi.com/documentation/current/examples/ |
| Full API reference | https://www.gurobi.com/documentation/current/refman/py_python_api_details.html |
| Academic license (free, unlimited) | https://www.gurobi.com/academia/academic-program-and-licenses/ |
| Community Q&A | https://support.gurobi.com/hc/en-us/community/topics |

> All 6 modules complete. Every cell runs on the free Gurobi web license (<=2000 vars, <=2000 constraints).
